# Samples from models on 60km -> 2.2km-4x over Birmingham

In [ ]:
%reload_ext autoreload

%autoreload 2

%reload_ext dotenv
%dotenv

In [ ]:
from mlde_analysis.furflex_default_params import *

In [ ]:
import dask
import dask.array
from dask.distributed import Client
import hvplot.xarray
import IPython
import matplotlib
import matplotlib.pyplot as plt
import numpy as np

from IPython.display import HTML, display_html
from matplotlib import animation

from mlde_analysis.furflex_data import prep_eval_data
from mlde_analysis.examples import plot_examples, em_timestamps
from mlde_analysis import cp_model_rotated_pole, STYLES, plot_map

In [ ]:
client = Client()
client

In [ ]:
matplotlib.rcParams['figure.dpi'] = 100

In [ ]:
IPython.display.Markdown(desc)

In [ ]:
%reload_ext mlde_analysis.magics 
EVAL_DS, MODELS, CPM_DAS, PRED_DAS, VAR_DAS, MODELLABEL2SPEC = %load_eval_data
EVAL_DS

In [ ]:
def plot_frame(da, frame, ax, model_label):
    frame_da = da.isel(time=frame)
    # import pdb; pdb.set_trace()
    pr_quad = plot_map(frame_da, ax=ax, style="pr_hourly")
    ax.set_title(f"{model_label} {frame_da["time"].item()}")
    
    return pr_quad


interval=500
var="pr"


def animate_day(ds, example_spec, filepath = None):
    example = ds.sel(time=slice(*example_spec["times"]), **example_spec["query"]).isel(model=0, sample_id=0)
    
    subplot_kw = dict(projection=cp_model_rotated_pole)
    fig = plt.figure(figsize=(5.5, 5.5), layout="constrained")
    axd = fig.subplot_mosaic([["pred", "truth"]], subplot_kw=subplot_kw)

    pr_quads = []
    
    pr_quad = plot_frame(example[f"pred_{var}"], 0, axd["pred"], model_label=example["model"].item())
    pr_quads.append(pr_quad)
    
    pr_quad = plot_frame(example[f"target_{var}"], 0, axd["truth"], model_label="CPM")
    pr_quads.append(pr_quad)
    
    cb = fig.colorbar(
        pr_quads[0],
        ax=axd.values(),
        location="bottom",
        orientation="horizontal",
        shrink=0.8,
        extend="both",
    )
    cb.ax.tick_params(axis="both", which="major", labelsize="small")
    cb.set_label("Precip [mm/hr]", fontsize="small")

    def update(frame):
        updated_pr_quads = []
    
        pr_quad = plot_frame(example[f"pred_{var}"], frame, axd[f"pred"], model_label=example["model"].item())
        updated_pr_quads.append(pr_quad)
    
        pr_quad = plot_frame(example[f"target_{var}"], frame, axd[f"truth"], model_label="CPM")
        updated_pr_quads.append(pr_quad)
        
        return updated_pr_quads

    nframes = example["time"].size
    anim = animation.FuncAnimation(fig, update, frames=nframes, blit=False, interval=interval)
    
    if filepath is not None:
        anim.save(filepath, 
                  writer='pillow', 
                  fps=int(1000/interval))
        print(f"Animation saved as '{filepath}'")
    
    plt.close(fig)  # prevent double display in notebook
    # Display as HTML5 video (works in JupyterLab)
    display_html(HTML(anim.to_html5_video()))

for source, examples in examples_to_plot.items():
    IPython.display.display_html(f"<h2>{source} Samples</h2>", raw=True)

    for label, example_spec in examples.items():
        animate_day(EVAL_DS[source], example_spec, filepath = f"{example_spec['times'][0]}-{'-'.join(example_spec['query'].values())}-hourly_precip.gif")